## Setup

Two-stage pipeline:
1. run LPCMCI once and save raw graphs in JSON (`causal_graphs_raw`),
2. search discriminative paths + deterministic classification from those JSON files (`causal_path_search`).


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from csi_vae_gumbel.causal_discovery_raw import run_raw_causal_discovery
from csi_vae_gumbel.path_search_from_raw import run_path_search_from_raw
from csi_vae_gumbel.settings import Settings

settings = Settings()


## Parameters

Use `MAX_WORKERS` to parallelize all activities in causal discovery.


In [ ]:
# Stage 1: raw causal discovery
IND_TEST = "parcorr"
TAU_MIN = 1
TAU_MAX = settings.test_window_size // settings.train_window_size - 1
PC_ALPHA = 1e-3
STRIDE = settings.train_window_size
USE_FULL_ONEHOT = True
MAX_WORKERS = 12

# Stage 2: path search + deterministic classification
FILTER_THRESHOLD = 0.1
PATH_THRESHOLD = 0.0
MAX_EDGES_PER_PATH = 4
MIN_EDGES_PER_PATH = 2
TOP_PATHS_PER_ACTIVITY = 150
TOP_RULES_PER_ACTIVITY = 50
RULE_MIN_DELTA = 0.0
CLASSIFIER_STRIDE = 75
CLASSIFIER_TRAIN_RATIO = 0.7
SEGMENT_LENGTH = 64
SEGMENT_HOP = 12
MAX_SEGMENTS_PER_ACTIVITY = None
SAVE_FIGS = True


## 1) Run Raw LPCMCI


In [ ]:
raw_result = run_raw_causal_discovery(
    settings=settings,
    ind_test_name=IND_TEST,
    tau_min=TAU_MIN,
    tau_max=TAU_MAX,
    pc_alpha=PC_ALPHA,
    stride=STRIDE,
    use_full_onehot=USE_FULL_ONEHOT,
    max_workers=MAX_WORKERS,
)
raw_result["output_dir"]


## 2) Search Paths + Classify From Raw Graphs


In [ ]:
path_result = run_path_search_from_raw(
    settings=settings,
    raw_dir=Path(raw_result["output_dir"]),
    filter_threshold=FILTER_THRESHOLD,
    path_threshold=PATH_THRESHOLD,
    max_edges_per_path=MAX_EDGES_PER_PATH,
    min_edges_per_path=MIN_EDGES_PER_PATH,
    top_paths_per_activity=TOP_PATHS_PER_ACTIVITY,
    top_rules_per_activity=TOP_RULES_PER_ACTIVITY,
    rule_min_delta=RULE_MIN_DELTA,
    classifier_stride=CLASSIFIER_STRIDE,
    classifier_train_ratio=CLASSIFIER_TRAIN_RATIO,
    segment_length=SEGMENT_LENGTH,
    segment_hop=SEGMENT_HOP,
    max_segments_per_activity=MAX_SEGMENTS_PER_ACTIVITY,
    save_figs=SAVE_FIGS,
)
path_result["output_dir"]


## Inspect


In [ ]:
output_dir = Path(path_result["output_dir"])
metrics = json.loads((output_dir / "deterministic_classifier_metrics.json").read_text())
summary = json.loads((output_dir / "graphs_summary.json").read_text())
rules = json.loads((output_dir / "classification_rules.json").read_text())
print("n_latent_variables:", summary["n_latent_variables"])
print("n_segments:", metrics["n_segments"])
print("accuracy:", metrics["accuracy"])

for activity, activity_rules in rules["activities"].items():
    print(f"\n[{activity}] {len(activity_rules)} rules")
    for rule in activity_rules[:3]:
        print(" -", rule["rule_text"])
